In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# Set default styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100


In [4]:
def plot_rfm_distributions(rfm_df):
    """Plot distributions of RFM metrics"""
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    sns.histplot(rfm_df['Recency'], kde=True)
    plt.title('Distribution of Recency (days)')
    plt.xlabel('Recency (days)')

    plt.subplot(1, 3, 2)
    sns.histplot(rfm_df['Frequency'], kde=True)
    plt.title('Distribution of Frequency')
    plt.xlabel('Frequency')
    plt.xlim(0, rfm_df['Frequency'].quantile(0.95))

    plt.subplot(1, 3, 3)
    sns.histplot(rfm_df['Monetary'], kde=True)
    plt.title('Distribution of Monetary Value')
    plt.xlabel('Monetary Value')
    plt.xlim(0, rfm_df['Monetary'].quantile(0.95))

    plt.tight_layout()
    plt.show()


In [5]:
def plot_segment_distribution(df, segment_column='Segment'):
    """Plot distribution of customer segments"""
    segment_counts = df[segment_column].value_counts().reset_index()
    segment_counts.columns = ['Segment', 'Count']
    segment_counts['Percentage'] = segment_counts['Count'] / segment_counts['Count'].sum() * 100

    plt.figure(figsize=(14, 8))
    sns.barplot(x='Count', y='Segment', data=segment_counts.sort_values('Count', ascending=False))
    plt.title('Number of Customers by Segment')
    plt.xlabel('Number of Customers')
    plt.tight_layout()
    plt.show()

    # Create a treemap visualization
    fig = px.treemap(segment_counts,
                    path=['Segment'],
                    values='Count',
                    color='Count',
                    hover_data=['Percentage'],
                    color_continuous_scale='Viridis')

    fig.update_layout(title='Customer Segments Treemap')
    fig.show()


In [6]:
def plot_radar_chart(df, metrics, group_column):
    """Create a radar chart to compare segments across multiple metrics"""
    # Get unique groups
    groups = df[group_column].unique()

    # Create a figure
    fig = go.Figure()

    # Add a trace for each group
    for group in groups:
        group_data = df[df[group_column] == group]

        # Calculate the average for each metric
        values = [group_data[metric].mean() for metric in metrics]

        # Add the first value again to close the polygon
        values.append(values[0])
        metric_names = metrics + [metrics[0]]

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=metric_names,
            fill='toself',
            name=group
        ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )
        ),
        title="Comparison by " + group_column
    )

    fig.show()

In [7]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    """Plot a confusion matrix"""
    from sklearn.metrics import confusion_matrix

    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks([0.5, 1.5], ['Active', 'Churned'])
    plt.yticks([0.5, 1.5], ['Active', 'Churned'])
    plt.show()


In [8]:
def plot_feature_importance(feature_names, importance_values, title='Feature Importance'):
    """Plot feature importance"""
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance_values
    })
    importance_df = importance_df.sort_values('Importance', ascending=False)

    plt.figure(figsize=(12, 6))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [9]:
def plot_roc_curves(models_dict, X_test, y_test):
    """Plot ROC curves for multiple models"""
    from sklearn.metrics import roc_curve, auc

    plt.figure(figsize=(10, 8))

    for model_name, model in models_dict.items():
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()


In [10]:
def plot_value_risk_matrix(df, value_column, risk_column):
    """Plot a value-risk matrix"""
    risk_matrix = pd.crosstab(
        df[value_column],
        df[risk_column],
        values=df['CustomerID'],
        aggfunc='count'
    )

    plt.figure(figsize=(10, 6))
    sns.heatmap(risk_matrix, annot=True, fmt='d', cmap='YlGnBu')
    plt.title(f'{value_column}-{risk_column} Matrix')
    plt.tight_layout()
    plt.show()